In [ ]:
import tensorflow as tf

from tensorflow.keras.layers import Dense, Input, Flatten, Conv2D, GlobalMaxPooling2D, MaxPooling2D, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import cv2
import imghdr

from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
from google.colab import files
files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
!mkdir -p ~/.kaggle

In [ ]:
!cp kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!pwd

In [ ]:
!kaggle datasets download -d yousefmohamed20/sentiment-images-classifier

In [ ]:
!unzip sentiment-images-classifier.zip

In [ ]:
!ls '6 Emotions for image classification'

In [ ]:
plt.imshow(image.load_img('6 Emotions for image classification/anger/white-woman-angry-behind-wheel-of-car-in-daylight-shutterstock___22114439556.jpg'), cmap='gray');

In [ ]:
import random, shutil, sys, os

In [ ]:
'''# splitting file

images_files.sort() # makes sure that the files have a fixed order before shuffling
random.seed(230)
random.shuffle(image_files) # shuffles the ordering of filenames (deterministic given the chosen seed)

split_data = int(0.33*len(image_files))
train_data = images_files[:split_data]
test_data = images_files[split_data:]'''


In [ ]:
data_dir = '6 Emotions for image classification'

In [ ]:
# removing not supported images

img_exts = ['jpeg', 'jpg', 'bmp', 'png']

for image_class in os.listdir(data_dir):
  for image in os.listdir(os.path.join(data_dir, image_class)):
    image_path = os.path.join(data_dir, image_class, image)
    try:
      img = cv2.imread(image_path)
      tip = imghdr.what(image_path)
      if tip not in img_exts:
        print('image not in ext list {}'.format(image_path))
        os.remove(image_path)
    except Exception as e:
      print('issue with image {}'.format(image_path))


In [ ]:
data = tf.keras.utils.image_dataset_from_directory(data_dir, batch_size=128)

In [ ]:
data_iterator = data.as_numpy_iterator()

In [ ]:
batch = data_iterator.next()

In [ ]:
fig, ax = plt.subplots(ncols=4, figsize = (20,20))
for idx, img in enumerate(batch[0][:4]):
  ax[idx].imshow(img.astype(int))
  ax[idx].title.set_text(batch[1][idx]);

In [ ]:
!ls '6 Emotions for image classification'

In [ ]:
# scaling the data

data = data.map(lambda x, y: (x/255, y))

In [ ]:
data.as_numpy_iterator().next()

In [ ]:
print(f' scaled data max: {data.as_numpy_iterator().next()[0].max()}')
print(f' scaled data min: {data.as_numpy_iterator().next()[0].min()}')

In [ ]:
# split data

train_size = int(len(data)*.7)
val_size = int(len(data)*.2) + 1
test_size = int(len(data)*.1)+1 # used post evaluation

In [ ]:
print(f' data size: {len(data)}')
print(f' train size: {(train_size)}')
print(f' test size: {(test_size)}')
print(f' val size: {(val_size)}')

In [ ]:
train = data.take(train_size)
val = data.skip(train_size).take(val_size)
test = data.skip(train_size+val_size).take(test_size)

In [ ]:
image_size = [256,256,3]
classes = 6


i = Input(shape=(image_size))
x = Conv2D(16, (3,3), activation = 'relu')(i)
x = MaxPooling2D()(x)
x = Conv2D(32, (3,3), activation = 'relu')(x)
x = Conv2D(32, (3,3), activation = 'relu')(x)
x = MaxPooling2D()(x)
x = Conv2D(64, (3,3), activation = 'relu')(x)
x = GlobalMaxPooling2D()(x)
x = Dense(256, activation = 'relu')(x)
x = Dense(classes, activation = 'softmax')(x)

model = Model(i,x)


In [ ]:
model.compile(
    loss = 'sparse_categorical_crossentropy',
    optimizer = Adam(learning_rate = 0.001, ema_momentum=0.89),
    metrics = ['accuracy']
)

In [ ]:
r = model.fit(train, validation_data = (val), epochs = 105)

In [ ]:
plt.plot(r.history['loss'], label = 'train')
plt.plot(r.history['val_loss'], label='test')
plt.legend();

In [ ]:
plt.plot(r.history['accuracy'], label='train')
plt.plot(r.history['val_accuracy'], label = 'test')
plt.legend();

In [ ]:
from tensorflow.keras.metrics import Precision, Recall, CategoricalAccuracy

In [ ]:
pre = Precision()
rec = Recall()
acc = CategoricalAccuracy()

In [ ]:
for batch in test.as_numpy_iterator():
  X, y = batch
  yhat = model.predict(X).argmax(axis=1)
  pre.update_state(y, yhat)
  recc.update_state(y,yhat)
  acc.update_state(y,yhat)




In [ ]:
y.shape